# CodePause Phase 2: Quality & Scale

This notebook implements the Phase 2 workflow focusing on solving the evaluation gap and scaling with QLoRA on Qwen 1.5B.

## 1. Setup Environment

In [ ]:
!pip install -q transformers peft bitsandbytes datasets accelerate trl scipy pytest pytest-cov pyyaml

import os
import subprocess
import sys
import time

REPO_URL = "https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git"
BRANCH = "main"
REPO_DIR = "/content/amdh"

if not os.path.exists(f"{REPO_DIR}/.git"):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}

%cd {REPO_DIR}
!git log --oneline -1


def run_step(name, cmd, cwd=None):
    print(f"\n=== {name} - START ===", flush=True)
    print(" ".join(cmd), flush=True)
    start = time.time()
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in proc.stdout:
        output_lines.append(line)
        print(line, end="", flush=True)
        sys.stdout.flush()
    return_code = proc.wait()
    elapsed = time.time() - start
    print(f"\n=== {name} - END ({elapsed:.1f}s, rc={return_code}) ===", flush=True)
    if return_code != 0:
        tail = "".join(output_lines[-40:])
        raise RuntimeError(f"HARD STOP: {name} failed with exit code {return_code}\n{tail}")
    return "".join(output_lines)


## 2. Sanity Baseline Evaluation
Verify the baseline performance on the 5-task sanity dataset.

In [ ]:
run_step(
    "Baseline sanity evaluation",
    [
        "python",
        "eval/evaluate_baseline.py",
        "--model_name",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--problems_path",
        "data/sanity_problems.jsonl",
        "--output_path",
        "results/phase2_sanity_baseline.jsonl",
        "--prompt_template",
        "baseline_qwen_instruct",
    ],
)


## 3. Adapter Probing
Test adapter attachment and output divergence.

In [ ]:
run_step(
    "Adapter probe",
    [
        "python",
        "eval/adapter_probe.py",
        "--adapter_path",
        "outputs/phase2_recipe_a",
    ],
)


## 4. Fine-Tuning (Recipe A)
Execute training using the standard QLoRA recipe.

In [ ]:
run_step(
    "Recipe A QLoRA training",
    [
        "python",
        "training/sft_lora.py",
        "--model_name",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--dataset_path",
        "data/thinkanywhere_sft_v2.jsonl",
        "--output_dir",
        "outputs/phase2_recipe_a",
        "--load_in_4bit",
        "--bnb_4bit_compute_dtype",
        "float16",
        "--bnb_4bit_quant_type",
        "nf4",
        "--max_steps",
        "100",
        "--limit_samples",
        "100",
        "--max_seq_length",
        "1024",
        "--lora_rank",
        "8",
        "--lora_alpha",
        "32",
        "--lora_dropout",
        "0.05",
        "--learning_rate",
        "2e-4",
        "--per_device_train_batch_size",
        "1",
        "--gradient_accumulation_steps",
        "4",
    ],
)


## 5. Evaluation & Reporting
Compare fine-tuned vs baseline.

In [ ]:
import os

ADAPTER_PATH = "outputs/phase2_recipe_a"

if not os.path.exists(f"{ADAPTER_PATH}/adapter_config.json"):
    raise RuntimeError(
        f"HARD STOP: adapter_config.json not found at {ADAPTER_PATH}. "
        "Run Recipe A training first and verify it completed successfully."
    )

run_step(
    "Fine-tuned sanity evaluation",
    [
        "python",
        "eval/evaluate_finetuned.py",
        "--base_model",
        "Qwen/Qwen1.5-1.8B-Chat",
        "--adapter_path",
        ADAPTER_PATH,
        "--problems_path",
        "data/sanity_problems.jsonl",
        "--output_path",
        "results/phase2_sanity_finetuned.jsonl",
        "--prompt_template",
        "baseline_qwen_instruct",
    ],
)


## 6. Drive Export
Persist results to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs/phase2_recipe_a /content/drive/MyDrive/codepause_phase2/